# Lab 20: XAI and Deployment

**Module 20** | Read `notes/20-shap-lime-xai-deployment.pdf` before running this notebook.

Train a fraud classifier, explain predictions with SHAP and LIME, and expose a Flask /predict endpoint.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## Explainable AI and Model Deployment

Machine learning models used for fraud detection must be **interpretable** and **deployable**.
This lab covers three layers:

1. **Train** a scikit-learn Random Forest on the credit-fraud sample (V1 to V28 features).
2. **Explain** predictions with **SHAP** (tree attributions) and **LIME** (local linear approximation).
3. **Serve** the model through a minimal **Flask** REST API at `POST /predict`.

The Flask app is tested with Flask's **test client** so the notebook does not block waiting for a server process.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **Explainable AI (XAI)** | Methods to explain why a model made a particular prediction. Critical for trust, debugging, and regulatory compliance in finance. |
| **SHAP (SHapley Additive exPlanations)** | Assigns each feature a contribution (Shapley value) to the prediction. Positive SHAP for fraud class means that feature pushed the score toward fraud. |
| **Shapley value** | From game theory: fair way to divide credit among features. Sum of SHAP values equals the model output (minus baseline). |
| **TreeExplainer** | SHAP algorithm optimized for tree models (Random Forest, XGBoost). Computes exact Shapley values efficiently for trees. |
| **LIME (Local Interpretable Model-agnostic Explanations)** | Explains one prediction by training a simple local model (linear) near that data point. Feature weights show which inputs mattered for *this* transaction. |
| **Flask** | A lightweight Python web framework. Here it wraps the model in HTTP endpoints so other applications can send JSON and get predictions back. |
| **REST API** | An HTTP interface: clients send requests (GET/POST) and receive JSON responses. `POST /predict` accepts feature rows and returns class labels and probabilities. |
| **Test client** | Flask utility that simulates HTTP requests in-process. Lets you test endpoints without starting a separate server. |

Refer back to this table whenever a term appears in code or output.


### Train the fraud classifier

**What this cell does:** Loads credit-fraud data, trains a Random Forest, prints evaluation metrics, and saves the model to disk.

**Why we do this:** You need a trained model before explaining or deploying it. Stratified splitting keeps the rare fraud class represented in both train and test sets.


**What to look for in the output**

You should see:
- Loaded file name and row count
- Confusion matrix counts (most predictions on the legit class)
- classification_report with precision/recall for legit and fraud
- `Saved model bundle to .../models/credit_fraud_rf.joblib`


In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from runtime_env import configure_runtime

configure_runtime(quiet=True)

import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

parquet_path = ROOT / "datasets" / "credit_fraud_sample.parquet"
csv_path = ROOT / "datasets" / "credit_fraud_sample.csv"
if parquet_path.exists():
    df = pd.read_parquet(parquet_path)
    print(f"Loaded {parquet_path.name}: {len(df):,} rows")
else:
    df = pd.read_csv(csv_path)
    print(f"Loaded {csv_path.name}: {len(df):,} rows")

feature_cols = [f"V{i}" for i in range(1, 29)]
X = df[feature_cols].values
y = df["Class"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
print()
print("Confusion matrix counts (rows=actual, cols=predicted):")
print(f"              pred 0    pred 1")
print(f"  actual 0    {cm[0, 0]:6d}    {cm[0, 1]:6d}")
print(f"  actual 1    {cm[1, 0]:6d}    {cm[1, 1]:6d}")
print()
print(classification_report(y_test, y_pred, target_names=["legit", "fraud"]))

# Save model + feature column names together so the API knows input order
model_path = ROOT / "models" / "credit_fraud_rf.joblib"
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump({"model": clf, "feature_cols": feature_cols}, model_path)
print(f"Saved model bundle to {model_path}")


### SHAP: global tree attributions

**What this cell does:** Uses `shap.TreeExplainer` to compute SHAP values for one test transaction and lists the top 10 features.

**Why we do this:** SHAP answers "which features pushed this prediction toward fraud or legit?"
**What SHAP values mean:** each feature gets a number showing its contribution to the model output for the fraud class. Positive = pushes toward fraud; negative = pushes toward legit. They sum (with a baseline) to the model's output.


**What to look for in the output**

You should see:
- SHAP values header for test row 0
- True label and predicted prob(fraud)
- Top 10 features ranked by absolute SHAP value with direction (toward fraud or toward legit)


In [ ]:
import shap

# TreeExplainer uses the tree structure for exact, fast SHAP values
explainer = shap.TreeExplainer(clf)
explain_row = 0
shap_values = explainer.shap_values(X_test[explain_row : explain_row + 1])

# For binary classification, shap_values may be a list [class0, class1]
if isinstance(shap_values, list):
    fraud_shap = np.asarray(shap_values[1]).reshape(-1)
    print(f"SHAP values for test row {explain_row} (fraud class, positive = pushes toward fraud)")
else:
    fraud_shap = np.asarray(shap_values).reshape(-1)
    print(f"SHAP values for test row {explain_row}")

true_label = int(y_test[explain_row])
fraud_prob = float(clf.predict_proba(X_test[explain_row : explain_row + 1])[0][1])
print(f"True label: {true_label} ({'fraud' if true_label == 1 else 'legit'})")
print(f"Predicted prob(fraud): {fraud_prob:.4f}")
print()
print("Top 10 features by |SHAP|:")
ranked = sorted(
    zip(feature_cols, fraud_shap),
    key=lambda x: abs(float(x[1])),
    reverse=True,
)
for rank, (feat, val) in enumerate(ranked[:10], start=1):
    v = float(val)
    direction = "toward fraud" if v > 0 else "toward legit"
    print(f"  {rank:2d}. {feat}: {v:+.5f}  ({direction})")


### LIME: local interpretable explanation

**What this cell does:** Uses LIME to explain the same test row with a local linear model.

**Why we do this:** LIME is model-agnostic and produces human-readable feature weights. Compare LIME's top features to SHAP's: they often agree on the most important inputs but can differ on minor ones.


**What to look for in the output**

You should see:
- LIME explanation header for the same test row
- True label and predicted prob(fraud)
- A list of feature conditions with signed weights (positive = pushes toward that class)


In [ ]:
import lime.lime_tabular

# LIME needs training data distribution to generate perturbed samples nearby
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    X_train,
    feature_names=feature_cols,
    class_names=["legit", "fraud"],
    mode="classification",
    random_state=42,
)

lime_row = explain_row
# explain_instance perturbs the input, queries predict_proba, fits a sparse linear model
lime_exp = lime_explainer.explain_instance(
    X_test[lime_row],
    clf.predict_proba,
    num_features=10,
)

print(f"LIME explanation for test row {lime_row}")
print(f"True label: {int(y_test[lime_row])}, predicted prob(fraud): {fraud_prob:.4f}")
print()
print("LIME feature list (as returned by as_list()):")
for feat, weight in lime_exp.as_list():
    print(f"  {feat}: {weight:+.4f}")


### Flask inference API

**What this cell does:** Defines a Flask app with `GET /health` and `POST /predict`, then tests both endpoints in-process.

**Why we do this:** In production, other systems call your model over HTTP instead of importing Python code.
**What each endpoint does:**
- `GET /health`: returns status ok and feature count (used by load balancers and monitoring)
- `POST /predict`: accepts JSON `{"instances": [[V1,..,V28], ...]}`, runs `predict_proba`, returns labels and probabilities


**What to look for in the output**

You should see three HTTP responses pretty-printed as JSON:
- GET /health with status ok and features count 28
- POST /predict with predictions and probabilities for 3 test rows
- POST /predict with an error message for invalid feature count (400 status)


In [ ]:
import json

from flask import Flask, jsonify, request

# root_path lets Flask initialize when this cell is exec'd outside a normal script file
app = Flask(__name__, root_path=str(ROOT))
# Load the saved model bundle at startup (same pattern as a production server)
_bundle = joblib.load(model_path)
_model = _bundle["model"]
_feature_cols = _bundle["feature_cols"]


@app.route("/health", methods=["GET"])
def health():
    # Simple liveness check: confirms the app and model loaded correctly
    return jsonify({"status": "ok", "features": len(_feature_cols)})


@app.route("/predict", methods=["POST"])
def predict():
    # Parse JSON body; force=True accepts requests without Content-Type header
    payload = request.get_json(force=True, silent=True) or {}
    rows = payload.get("instances")
    if not rows:
        return jsonify({"error": "Provide JSON: {'instances': [[V1,..,V28], ...]}"}), 400

    X_in = np.asarray(rows, dtype=float)
    if X_in.ndim == 1:
        X_in = X_in.reshape(1, -1)  # allow a single flat row
    if X_in.shape[1] != len(_feature_cols):
        return jsonify({
            "error": f"Expected {len(_feature_cols)} features per row, got {X_in.shape[1]}"
        }), 400

    probs = _model.predict_proba(X_in)
    preds = _model.predict(X_in)
    return jsonify({
        "predictions": preds.astype(int).tolist(),
        "probabilities": probs.tolist(),
        "feature_cols": _feature_cols,
    })


def pprint_response(label: str, response) -> None:
    print(f"{label} (HTTP {response.status_code}):")
    try:
        body = response.get_json()
        print(json.dumps(body, indent=2))
    except Exception:
        print(response.data.decode())
    print()


# test_client simulates HTTP without binding a port or blocking the notebook
with app.test_client() as client:
    health_resp = client.get("/health")
    pprint_response("GET /health", health_resp)

    test_rows = X_test[:3].tolist()
    pred_resp = client.post("/predict", json={"instances": test_rows})
    pprint_response("POST /predict (3 test rows)", pred_resp)

    bad_resp = client.post("/predict", json={"instances": [[1.0, 2.0]]})
    pprint_response("POST /predict (invalid feature count)", bad_resp)


### Prediction summary on held-out test rows

**What this cell does:** Prints a table of true labels, predictions, and fraud probabilities for the first 8 test rows.

**Why we do this:** A quick sanity check that the classifier behaves reasonably before you deploy or explain more rows.


**What to look for in the output**

You should see:
- A table with Row, True, Pred, P(fraud), and Match columns for up to 8 rows
- Overall test accuracy printed at the bottom (typically high, but fraud recall may be lower due to imbalance)


In [ ]:
probs_all = clf.predict_proba(X_test)
preds_all = clf.predict(X_test)

print("=" * 58)
print("TEST ROW PREDICTION SUMMARY (first 8 rows)")
print("=" * 58)
print(f"{'Row':>4} | {'True':>4} | {'Pred':>4} | {'P(fraud)':>10} | {'Match':>5}")
print("-" * 58)
for i in range(min(8, len(X_test))):
    true_l = int(y_test[i])
    pred_l = int(preds_all[i])
    p_fraud = probs_all[i][1]
    match = "yes" if true_l == pred_l else "no"
    print(f"{i:>4} | {true_l:>4} | {pred_l:>4} | {p_fraud:>10.4f} | {match:>5}")

correct = int((preds_all == y_test).sum())
print("-" * 58)
print(f"Overall test accuracy: {correct}/{len(y_test)} = {correct / len(y_test):.4f}")


## Summary

| Component | Purpose |
|-----------|---------|
| Random Forest | Strong baseline on tabular fraud data |
| SHAP TreeExplainer | Exact Shapley values for tree models (top features for one row) |
| LIME | Human-readable local explanation for a single transaction |
| Flask `/predict` | Production-style HTTP inference (tested in-process here) |

**Next steps:** containerize the Flask app (Docker), add input validation (pydantic), and log prediction latency plus explanation metadata to MLflow for audit trails.
